In [1]:
import torch
import torch.nn as nn

class LateFeatureGate(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(c, c, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        g = self.gate(x)
        self.last_gate = g.detach()   # critical for logging
        return x * g


In [2]:
import torch
import torch.nn as nn
from ultralytics.nn.modules.block import C3k2
import ultralytics.nn.tasks as yolo_tasks
from ultralytics.utils.ops import make_divisible

class C3k2WithLateGate(C3k2):
    def __init__(self, *args, **kwargs):
        # args[0] is usually c1 (injected by parser)
        # args[1] is usually c2 (from your YAML [512])
        
        # 1. Flexible Argument Extraction
        if len(args) >= 2:
            c1, c2 = args[0], args[1]
            remaining_args = args[2:]
        else:
            # Fallback for weird parsing edge cases
            c1 = args[0]
            c2 = c1
            remaining_args = args[1:]

        # 3. Initialize Parent with the standard YOLO11 signature
        # We map the extracted args back to the named params C3k2 expects
        super().__init__(
            c1=int(c1), 
            c2=int(c2), 
            n=int(remaining_args[0]) if len(remaining_args) > 0 else 1,
            shortcut=True # Standard default
        )
        
        # 4. Final safety check on Gate dimensions
        c_out = self.cv2.conv.out_channels
        
        self.late_gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(c2, c2, 1, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        y = super().forward(x)
        return y *self.late_gate(y)

In [3]:
import ultralytics.nn.tasks as yolo_tasks
from ultralytics.utils.ops import make_divisible

# We override the global module search
yolo_tasks.C3k2WithLateGate = C3k2WithLateGate
globals()["C3k2WithLateGate"] = C3k2WithLateGate

# This is the secret: We tell the parser how to calculate channels for this block
# If we are in 'n' scale, width=0.25. 512 * 0.25 = 128.
def custom_parse_model(d, ch, verbose=True):
    # This is a wrapper around the original parse_model to catch our module
    # But a cleaner way is to just use the YAML properly with the scaled value
    return yolo_tasks.parse_model(d, ch, verbose)

In [4]:
from ultralytics import YOLO

yolo = YOLO("yolo11n_lategated.yaml")
yolo.model.info()



YOLO11n_lategated summary: 175 layers, 2,631,952 parameters, 2,631,936 gradients, 6.6 GFLOPs


(175, 2631952, 2631936, 6.5996928)

In [5]:
# import torch
# import torch.nn as nn
# from ultralytics.nn.modules.conv import Conv

# class FirstConvAdapter(nn.Module):
#     def __init__(self, orig):
#             super().__init__()

#             self.proj = Conv(4, 3, k=1, s=1, p=0, act=True)
#             self.conv = orig

#             # ---- SAFE graph metadata transfer ----
#             self.f = getattr(orig, "f", -1)
#             self.i = getattr(orig, "i", 0)
#             self.type = getattr(orig, "type", self.__class__.__name__)
#             self.np = getattr(orig, "np", sum(p.numel() for p in self.parameters()))

#     def forward(self, x):
#         if x.shape[1] == 4:
#             x = self.proj(x)
#         return self.conv(x)



In [6]:
# m0 = yolo.model.model[0]
# yolo.model.model[0] = FirstConvAdapter(m0)

# print("✅ First conv adapted with graph metadata preserved")


In [7]:
import torch

device = torch.device("cuda:0")

yolo.model = yolo.model.to(device)
yolo.model.eval()


DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C3k2(
      (cv1): Conv(
        (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
   

In [8]:
# device = torch.device("cuda:0")
# yolo.model = yolo.model.to(device)
# yolo.model.eval()

# x = torch.randn(1, 4, 1024, 1024, device=device)
# with torch.no_grad():
#     _ = yolo.model(x)

# print("✅ Forward OK — 4ch input + late gating working")


In [9]:
from ultralytics.utils import LOGGER

gate_stats = []

def gate_hook(module, inp, out):
    if hasattr(module, "last_gate"):
        g = module.last_gate
        gate_stats.append((
            g.mean().item(),
            g.std().item()
        ))

for m in yolo.model.modules():
    if isinstance(m, LateFeatureGate):
        m.register_forward_hook(gate_hook)


In [10]:
def on_train_epoch_end(trainer):
    if not gate_stats:
        return

    means = [m for m, _ in gate_stats]
    stds  = [s for _, s in gate_stats]

    LOGGER.info(
        f"[LATE GATE] Epoch {trainer.epoch}: "
        f"mean={sum(means)/len(means):.4f}, "
        f"std={sum(stds)/len(stds):.4f}"
    )

    gate_stats.clear()

yolo.add_callback("on_train_epoch_end", on_train_epoch_end)


In [11]:
yolo.train(
    data="weighted_fusion.yaml",
    imgsz=1024,
    epochs=500,
    patience=40,
    batch=8,
    device=0,
    optimizer="AdamW",
    lr0=3e-4,
    cos_lr=True,
    warmup_epochs=5,
    amp=True,
    augment=False,
    project="weighted_late_gate_experiments",
    name="weighted_yolo11n_late_gate",
    workers=0
)


New https://pypi.org/project/ultralytics/8.4.13 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.6  Python-3.11.0 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5050 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=weighted_fusion.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0003, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n_lategated.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=weigh

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001CA4C0EC650>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [12]:
print(yolo.model.model[4])


C3k2(
  (cv1): Conv(
    (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
    (act): SiLU(inplace=True)
  )
  (cv2): Conv(
    (conv): Conv2d(96, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
    (act): SiLU(inplace=True)
  )
  (m): ModuleList(
    (0): Bottleneck(
      (cv1): Conv(
        (conv): Conv2d(32, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
    )
  )
)


In [13]:
sum(p.numel() for p in yolo.model.parameters())


2597907